# CNN Training Pipeline

End-to-end notebook covering every CNN process in `models/CNN/`:

0. Setup
1. Baseline student training (one or all)
2. Teacher training (one or all)
3. Knowledge distillation (one combo or all 15)
4. Optuna HPO (one student or all)
5. Optuna HPO with knowledge distillation (one combo or all 15)
6. Inference / sanity-check on a saved model
7. Evaluation via the team-standard `evaluation/` module (one model or all)

Each section has a small **Configure** cell at the top (constants to change) and a **Run** cell underneath. Re-run cells freely — outputs land in `models/CNN/checkpoints/` and `models/CNN/evaluations/`.

## 0. Setup

Adds the repo root and the CNN folder to `sys.path` (so peer modules like `datasets.loaders` and `evaluation.metrics` import cleanly), switches the working directory to the repo root, and loads everything we'll need.

In [1]:
import sys, os, copy, json, time
from pathlib import Path

_here = Path.cwd()
while not (_here / "datasets").exists() and _here != _here.parent:
    _here = _here.parent
REPO_ROOT = _here
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "models" / "CNN"))

import torch
import torch.nn as nn
import numpy as np

from main import Config, set_seed, get_loaders, _summary_metrics, train_student, train_teacher, train_student_with_kd
from students import CNN_Nano, CNN_Micro, CNN_Base, CNN_Large, CNN_XLarge
from teachers import ResNet50_1D, ResNet101_1D, ResNet152_1D
from training import train_model, evaluate
from distillation import distillation_loss
from evaluation.metrics import compute_multilabel_metrics, compute_curves
from evaluation.plots import save_metric_curves, save_confusion_matrices

STUDENT_CLS = {"nano": CNN_Nano, "micro": CNN_Micro, "base": CNN_Base, "large": CNN_Large, "xlarge": CNN_XLarge}
TEACHER_CLS = {"resnet50": ResNet50_1D, "resnet101": ResNet101_1D, "resnet152": ResNet152_1D}
STUDENT_NAMES = list(STUDENT_CLS.keys())
TEACHER_NAMES = list(TEACHER_CLS.keys())

CKPT_DIR = Path("models/CNN/checkpoints")
EVAL_DIR = Path("models/CNN/evaluations")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
EVAL_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"repo root: {REPO_ROOT}")
print(f"device:    {DEVICE}")

repo root: c:\UTMIST\psr-pipeline
device:    cuda


## 1. Train students

Calls `train_student(cfg)` from `main.py`. Saves the best-val-acc checkpoint to `models/CNN/checkpoints/student_<name>.pth` and prints test-set metrics at the end.

### 1.1 Train one student

In [2]:
# === Configure ===
STUDENT = "xlarge"     # one of: nano, micro, base, large, xlarge
EPOCHS = 100
BATCH_SIZE = 32
LR = 1e-3

# === Run ===
cfg = Config(mode="student", student_name=STUDENT, epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, device=DEVICE)
train_student(cfg)

student=xlarge  params=1,619,781


train:   1%|          | 1/100 [00:01<02:02,  1.24s/it]

[1/100] train_loss=0.7426  val_loss=0.6808  val_acc=49.32%  val_f1=67.31


train:   2%|▏         | 2/100 [00:02<01:38,  1.00s/it]

[2/100] train_loss=0.6597  val_loss=0.8193  val_acc=54.56%  val_f1=66.56


...

[99/100] train_loss=0.0994  val_loss=1.5808  val_acc=61.32%  val_f1=69.06


train: 100%|██████████| 100/100 [01:45<00:00,  1.06s/it]

[100/100] train_loss=0.1008  val_loss=1.5708  val_acc=61.15%  val_f1=69.47


test  acc=0.591  f1_macro=0.701  auprc=0.808


CNN_XLarge(
  (in_norm): Identity()
  (proj): Conv1d(768, 128, kernel_size=(1,), stride=(1,))
  (bn0): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv1): Conv1d(128, 192, kernel_size=(5,), stride=(1,), padding=(2,))
  (bn1): BatchNorm1d(192, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv1d(192, 256, kernel_size=(5,), stride=(1,), padding=(2,))
  (bn2): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv3): Conv1d(256, 384, kernel_size=(3,), stride=(1,), padding=(1,))
  (bn3): BatchNorm1d(384, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv4): Conv1d(384, 512, kernel_size=(3,), stride=(1,), padding=(1,))
  (bn4): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool): AdaptiveMaxPool1d(output_size=2)
  (fc1): Linear(in_features=1024, out_features=256, bias=True)
  (dropout): Dropout(p=0.4, inplace=False)
  (fc2): Linear

### 1.2 Train all students (sequential)

In [ ]:
# === Configure ===
EPOCHS = 100
BATCH_SIZE = 32
LR = 1e-3

# === Run ===
for name in STUDENT_NAMES:
    print(f"\n========== {name.upper()} ==========")
    cfg = Config(mode="student", student_name=name, epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, device=DEVICE)
    train_student(cfg)

## 2. Train teachers

1D ResNet teachers. Saves to `models/CNN/checkpoints/teacher_<name>.pth`. Teachers are bigger than students — expect ~6 s/epoch on a modern GPU.

### 2.1 Train one teacher

In [12]:
# === Configure ===
TEACHER = "resnet50"  # resnet50 | resnet101 | resnet152
EPOCHS = 100
BATCH_SIZE = 32
LR = 1e-3

# === Run ===
cfg = Config(mode="teacher", teacher_name=TEACHER, epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, device=DEVICE)
train_teacher(cfg)

teacher=resnet50  params=16,309,637


train:   1%|          | 1/100 [00:04<08:00,  4.85s/it]

[1/100] train_loss=0.9560  val_loss=0.7978  val_acc=46.96%  val_f1=57.66


train:   2%|▏         | 2/100 [00:09<07:35,  4.65s/it]

[2/100] train_loss=0.8332  val_loss=0.8044  val_acc=46.11%  val_f1=56.51


...

[99/100] train_loss=0.0009  val_loss=2.8059  val_acc=46.45%  val_f1=57.96


train: 100%|██████████| 100/100 [09:50<00:00,  5.90s/it]

[100/100] train_loss=0.0004  val_loss=2.8488  val_acc=45.78%  val_f1=57.86


test  acc=0.462  f1_macro=0.592  auprc=0.610


ResNet50_1D(
  (in_norm): InstanceNorm1d(768, eps=1e-05, momentum=0.1, affine=True, track_running_stats=False)
  (conv1): Conv1d(768, 64, kernel_size=(7,), stride=(2,), padding=(3,), bias=False)
  (bn1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool1d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck1D(
      (conv1): Conv1d(64, 64, kernel_size=(1,), stride=(1,), bias=False)
      (bn1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(1,), bias=False)
      (bn2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv1d(64, 256, kernel_size=(1,), stride=(1,), bias=False)
      (bn3): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
     

### 2.2 Train all teachers (sequential)

In [9]:
# === Configure ===
EPOCHS = 150
BATCH_SIZE = 32
LR = 1e-4

# === Run ===
for name in TEACHER_NAMES:
    print(f"\n========== {name.upper()} ==========")
    cfg = Config(mode="teacher", teacher_name=name, epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, device=DEVICE)
    train_teacher(cfg)


========== RESNET50 ==========
teacher=resnet50  params=16,308,101


train:   1%|          | 1/150 [00:04<11:20,  4.57s/it]

[1/150] train_loss=0.8246  val_loss=0.8675  val_acc=43.75%  val_f1=63.98


train:   1%|▏         | 2/150 [00:08<10:38,  4.31s/it]

[2/150] train_loss=0.7672  val_loss=0.8425  val_acc=46.28%  val_f1=62.01


...

train:  99%|█████████▉| 149/150 [35:04<00:18, 18.29s/it]

[149/150] train_loss=0.0392  val_loss=1.7036  val_acc=54.56%  val_f1=66.39


train: 100%|██████████| 150/150 [35:27<00:00, 14.18s/it]

[150/150] train_loss=0.0450  val_loss=1.7229  val_acc=54.05%  val_f1=66.38


test  acc=0.588  f1_macro=0.698  auprc=0.787


## 3. Knowledge distillation

Single-step KD: student trained against teacher logits. If a `teacher_<name>.pth` exists in `checkpoints/`, it's loaded; otherwise the teacher is pretrained on the fly for `teacher_epochs`. Saves to `student_kd_<student>_from_<teacher>.pth`.

### 3.1 Train one student with one teacher

In [ ]:
# === Configure ===
STUDENT = "nano"
TEACHER = "resnet50"
EPOCHS = 100
TEACHER_PRETRAIN_EPOCHS = 30   # only used if teacher_<name>.pth is missing
KD_TEMPERATURE = 4.0
KD_ALPHA = 0.5
BATCH_SIZE = 32
LR = 1e-3

# === Run ===
teacher_ckpt = CKPT_DIR / f"teacher_{TEACHER}.pth"
cfg = Config(
    mode="student_kd", student_name=STUDENT, teacher_name=TEACHER,
    epochs=EPOCHS, teacher_epochs=TEACHER_PRETRAIN_EPOCHS,
    kd_temperature=KD_TEMPERATURE, kd_alpha=KD_ALPHA,
    batch_size=BATCH_SIZE, lr=LR, device=DEVICE,
    teacher_ckpt=str(teacher_ckpt) if teacher_ckpt.exists() else None,
)
train_student_with_kd(cfg)

### 3.2 Train all 15 student-teacher KD combinations

Loops over every `(student, teacher)` pair. If a teacher checkpoint isn't on disk yet, it's pretrained the first time it's needed — subsequent pairs that share the same teacher reuse the saved file.

In [ ]:
# === Configure ===
EPOCHS = 100
TEACHER_PRETRAIN_EPOCHS = 30
KD_TEMPERATURE = 4.0
KD_ALPHA = 0.5
BATCH_SIZE = 32
LR = 1e-3

# === Run ===
for teacher_name in TEACHER_NAMES:
    teacher_ckpt = CKPT_DIR / f"teacher_{teacher_name}.pth"
    for student_name in STUDENT_NAMES:
        print(f"\n========== {student_name.upper()} from {teacher_name.upper()} ==========")
        cfg = Config(
            mode="student_kd", student_name=student_name, teacher_name=teacher_name,
            epochs=EPOCHS, teacher_epochs=TEACHER_PRETRAIN_EPOCHS,
            kd_temperature=KD_TEMPERATURE, kd_alpha=KD_ALPHA,
            batch_size=BATCH_SIZE, lr=LR, device=DEVICE,
            teacher_ckpt=str(teacher_ckpt) if teacher_ckpt.exists() else None,
        )
        train_student_with_kd(cfg)

## 4. Optuna HPO (no KD)

Calls `optimization.run` which sweeps architecture + training hyperparams per student size. Each trial trains up to `trial_epochs` with patience-based early stopping; trials scored by `val_accuracy × speed_penalty` against a 1-epoch baseline.

Outputs per size: `optuna_<size>.pth` (with `arch_kwargs`) and `optuna_<size>.txt` (human-readable summary).

### 4.1 HPO for one student

In [ ]:
from optimization import HPOConfig, run as run_hpo

# === Configure ===
STUDENT = "nano"
N_TRIALS = 50
TRIAL_EPOCHS = 100
EARLY_STOP_PATIENCE = 5
EARLY_STOP_MIN_DELTA = 2.0

# === Run ===
hcfg = HPOConfig(
    sizes=[STUDENT], n_trials_per_size=N_TRIALS, trial_epochs=TRIAL_EPOCHS,
    early_stop_patience=EARLY_STOP_PATIENCE, early_stop_min_delta=EARLY_STOP_MIN_DELTA,
    device=DEVICE,
)
run_hpo(hcfg)

### 4.2 HPO for all students

In [2]:
from optimization import HPOConfig, run as run_hpo

# === Configure ===
N_TRIALS = 50
TRIAL_EPOCHS = 100

# === Run ===
hcfg = HPOConfig(n_trials_per_size=N_TRIALS, trial_epochs=TRIAL_EPOCHS, device=DEVICE)
run_hpo(hcfg)

c:\Users\markk\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



========== NANO ==========
[baseline] training default architecture for 1 epoch(s)...


[I 2026-06-09 16:40:40,498] A new study created in memory with name: no-name-ba3e4a4f-0aa9-4d65-8ed5-8e05fe188965


[baseline] latency=0.650ms  val_f1=72.03
[study] running 50 trials...


[I 2026-06-09 16:40:45,255] Trial 0 finished with value: 0.5702371259847699 and parameters: {'proj_ch': 24, 'conv1_ch': 48, 'conv1_k': 5, 'pool': 4, 'fc1': 128, 'dropout': 0.027898772130408367, 'pool_type': 'max', 'lr': 0.00023345864076016249, 'weight_decay': 0.0002267398652378039, 'batch_size': 128}. Best is trial 0 with value: 0.5702371259847699.
[I 2026-06-09 16:40:49,112] Trial 1 finished with value: 0.7335449457168579 and parameters: {'proj_ch': 48, 'conv1_ch': 48, 'conv1_k': 7, 'pool': 8, 'fc1': 96, 'dropout': 0.18789978831283782, 'pool_type': 'avg', 'lr': 0.00582938454299474, 'weight_decay': 1.8427970406864558e-06, 'batch_size': 64}. Best is trial 1 with value: 0.7335449457168579.
[I 2026-06-09 16:41:00,409] Trial 2 finished with value: 0.7113219499588013 and parameters: {'proj_ch': 16, 'conv1_ch': 48, 'conv1_k': 5, 'pool': 8, 'fc1': 64, 'dropout': 0.012711670057204728, 'pool_type': 'max', 'lr': 0.0015446089075047066, 'weight_decay': 8.178476574339548e-05, 'batch_size': 8}. Best

...

[done] score=0.7557  f1=75.57  latency=0.893ms  ratio=0.919x  -> optuna_large.pth

========== XLARGE ==========
[baseline] training default architecture for 1 epoch(s)...


[I 2026-06-09 17:24:01,661] A new study created in memory with name: no-name-f1a623e6-22ec-41b5-9711-58a2ca3e28b7


[baseline] latency=0.936ms  val_f1=67.31
[study] running 50 trials...


[I 2026-06-09 17:24:26,561] Trial 0 finished with value: 0.7364802360534668 and parameters: {'proj_ch': 96, 'conv1_ch': 192, 'conv1_k': 5, 'conv2_ch': 512, 'conv2_k': 7, 'conv3_ch': 640, 'conv3_k': 7, 'conv4_ch': 768, 'conv4_k': 3, 'pool': 4, 'fc1': 384, 'dropout': 0.1656305710884955, 'pool_type': 'max', 'lr': 0.00043664735929796326, 'weight_decay': 3.5856126103453987e-06, 'batch_size': 8}. Best is trial 0 with value: 0.7364802360534668.
[I 2026-06-09 17:24:42,322] Trial 1 finished with value: 0.03691672682762146 and parameters: {'proj_ch': 64, 'conv1_ch': 192, 'conv1_k': 7, 'conv2_ch': 192, 'conv2_k': 3, 'conv3_ch': 640, 'conv3_k': 3, 'conv4_ch': 768, 'conv4_k': 7, 'pool': 8, 'fc1': 192, 'dropout': 0.007857296421683563, 'pool_type': 'avg', 'lr': 0.0003355151022721483, 'weight_decay': 0.0005280796376895364, 'batch_size': 32}. Best is trial 0 with value: 0.7364802360534668.
[I 2026-06-09 17:24:48,872] Trial 2 finished with value: 0.037443441152572636 and parameters: {'proj_ch': 128, 'co

[done] score=0.7365  f1=73.65  latency=0.923ms  ratio=0.986x  -> optuna_xlarge.pth

========== SUMMARY ==========
size        score    val_f1   latency_ms  baseline_ms   ratio
nano       0.7400    74.00        0.472        0.650   0.727x
micro      0.7531    75.31        0.804        0.898   0.896x
base       0.7354    73.54        1.300        1.699   0.765x
large      0.7557    75.57        0.893        0.971   0.919x
xlarge     0.7365    73.65        0.923        0.936   0.986x


## 5. Optuna HPO with knowledge distillation

Same scoring shape (`val_accuracy × speed_penalty`) but the student is trained against teacher logits instead of plain BCE. Tunes architecture, `lr`, `weight_decay`, `batch_size`, `kd_temperature`, and `kd_alpha`. The teacher is loaded once per study (from `teacher_<name>.pth`, or pretrained on the fly if missing) and reused across trials.

Outputs per combo: `optuna_kd_<student>_from_<teacher>.pth` and `optuna_kd_<student>_from_<teacher>.txt`.

In [ ]:
# === KD HPO helper ===
# Reuses the search space, baseline timing, latency measurement, and penalty from optimization.py;
# differs only in that the per-trial loop trains against the teacher.
import optuna
from optimization import (SEARCH_SPACES, TRAIN_SPACE, suggest_arch, measure_baseline,
                          measure_latency, speed_penalty)

def _train_kd_trial(student, teacher, train_loader, val_loader, lr, wd, T, alpha,
                    epochs, device, trial, patience, min_delta):
    optimizer = torch.optim.Adam(student.parameters(), lr=lr, weight_decay=wd)
    eval_criterion = nn.BCEWithLogitsLoss()
    teacher.eval()
    best_acc, best_state, last_milestone, stall = -1.0, None, -float("inf"), 0
    epoch = 0
    for epoch in range(epochs):
        student.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            s_logits = student(x)
            with torch.no_grad():
                t_logits = teacher(x)
            loss = distillation_loss(s_logits, t_logits, y, T, alpha)
            loss.backward(); optimizer.step()
        val = evaluate(student, val_loader, eval_criterion, device)
        acc = val["accuracy"]
        if acc > best_acc:
            best_acc, best_state = acc, copy.deepcopy(student.state_dict())
        if acc >= last_milestone + min_delta:
            last_milestone, stall = acc, 0
        else:
            stall += 1
        if stall >= patience: break
        trial.report(acc, epoch)
        if trial.should_prune(): raise optuna.TrialPruned()
    return best_acc, best_state, epoch + 1

def _load_or_pretrain_teacher(teacher_name, pretrain_epochs, device):
    teacher = TEACHER_CLS[teacher_name]().to(device)
    ckpt = CKPT_DIR / f"teacher_{teacher_name}.pth"
    if ckpt.exists():
        teacher.load_state_dict(torch.load(ckpt, map_location=device))
        print(f"[teacher] loaded {ckpt}")
    else:
        print(f"[teacher] no checkpoint, pretraining {teacher_name} for {pretrain_epochs} epochs")
        cfg = Config(mode="teacher", teacher_name=teacher_name, epochs=pretrain_epochs, device=device)
        teacher = train_teacher(cfg).to(device)
    teacher.eval()
    return teacher

def kd_hpo(student_name, teacher_name, n_trials=30, trial_epochs=60,
           teacher_pretrain_epochs=30, patience=5, min_delta=2.0, seed=42):
    device = torch.device(DEVICE)
    teacher = _load_or_pretrain_teacher(teacher_name, teacher_pretrain_epochs, device)

    # Baseline timing: default student arch trained 1 epoch with KD (so reference is comparable)
    set_seed(seed)
    base_model = STUDENT_CLS[student_name]().to(device)
    train_loader, val_loader, _ = get_loaders(Config(batch_size=32, seed=seed, device=DEVICE))
    base_opt = torch.optim.Adam(base_model.parameters(), lr=1e-3, weight_decay=1e-4)
    base_model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        base_opt.zero_grad()
        with torch.no_grad(): t = teacher(x)
        loss = distillation_loss(base_model(x), t, y, 4.0, 0.5)
        loss.backward(); base_opt.step()
    base_latency = measure_latency(base_model, val_loader, device)
    base_val_acc = evaluate(base_model, val_loader, nn.BCEWithLogitsLoss(), device)["accuracy"]
    print(f"[baseline] latency={base_latency:.3f}ms  val_acc={base_val_acc:.2f}%")

    best = {"score": -1.0}
    def objective(trial):
        set_seed(seed + trial.number)
        arch = suggest_arch(trial, student_name)
        lr  = trial.suggest_float("lr",           *TRAIN_SPACE["lr"],           log=True)
        wd  = trial.suggest_float("weight_decay", *TRAIN_SPACE["weight_decay"], log=True)
        bs  = trial.suggest_categorical("batch_size", TRAIN_SPACE["batch_size"])
        T   = trial.suggest_float("kd_temperature", 1.0, 10.0)
        a   = trial.suggest_float("kd_alpha",        0.1, 0.9)

        tl, vl, _ = get_loaders(Config(batch_size=bs, seed=seed, device=DEVICE))
        student = STUDENT_CLS[student_name](**arch).to(device)
        best_acc, best_state, epochs_run = _train_kd_trial(
            student, teacher, tl, vl, lr, wd, T, a, trial_epochs, device, trial, patience, min_delta,
        )
        student.load_state_dict(best_state)
        latency = measure_latency(student, vl, device)
        ratio = latency / base_latency
        score = (best_acc / 100.0) * speed_penalty(ratio)
        train_kw = {"lr": lr, "weight_decay": wd, "batch_size": bs, "kd_temperature": T, "kd_alpha": a}
        if score > best["score"]:
            best.update(score=score, state=best_state, arch_kwargs=arch, train_kwargs=train_kw,
                        val_accuracy=best_acc, latency_ms=latency, ratio=ratio, epochs_run=epochs_run)
        return score

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=seed),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3),
    )
    study.optimize(objective, n_trials=n_trials)

    pth = CKPT_DIR / f"optuna_kd_{student_name}_from_{teacher_name}.pth"
    torch.save({"state_dict": best["state"], "arch_kwargs": best["arch_kwargs"]}, pth)
    txt = CKPT_DIR / f"optuna_kd_{student_name}_from_{teacher_name}.txt"
    arch_repr = ", ".join(f"{k}={v!r}" for k, v in best["arch_kwargs"].items())
    txt.write_text(
        f"=== optuna_kd_{student_name}_from_{teacher_name} ===\n"
        f"trials run:         {len(study.trials)}\n"
        f"best score:         {best['score']:.4f}\n"
        f"best val accuracy:  {best['val_accuracy']:.2f}%\n"
        f"inference latency:  {best['latency_ms']:.3f} ms (baseline {base_latency:.3f}, ratio {best['ratio']:.3f}x)\n"
        f"epochs trained:     {best['epochs_run']} / {trial_epochs}\n\n"
        f"Arch kwargs:\n" + "\n".join(f"  {k:<14} = {v}" for k, v in best["arch_kwargs"].items()) + "\n\n"
        f"Training kwargs:\n" + "\n".join(f"  {k:<14} = {v}" for k, v in best["train_kwargs"].items()) + "\n"
    )
    print(f"[done] score={best['score']:.4f} acc={best['val_accuracy']:.2f}% latency={best['latency_ms']:.3f}ms -> {pth.name}")
    return best

### 5.1 KD HPO for one student-teacher combo

In [ ]:
# === Configure ===
STUDENT = "nano"
TEACHER = "resnet50"
N_TRIALS = 30
TRIAL_EPOCHS = 60

# === Run ===
kd_hpo(STUDENT, TEACHER, n_trials=N_TRIALS, trial_epochs=TRIAL_EPOCHS)

### 5.2 KD HPO for all 15 student-teacher combos

Heavy: 15 studies × `N_TRIALS` trials × variable epochs (with early stop). Defaults below assume an overnight run. Cut `N_TRIALS` or `TRIAL_EPOCHS` for quicker exploration.

In [ ]:
# === Configure ===
N_TRIALS = 20
TRIAL_EPOCHS = 50

# === Run ===
for teacher_name in TEACHER_NAMES:
    for student_name in STUDENT_NAMES:
        print(f"\n========== KD HPO: {student_name.upper()} from {teacher_name.upper()} ==========")
        kd_hpo(student_name, teacher_name, n_trials=N_TRIALS, trial_epochs=TRIAL_EPOCHS)

## 6. Inference / sanity check

Pick any checkpoint in `models/CNN/checkpoints/` and verify it loads and produces sensible predictions on a test batch. Filename convention drives which model class is rebuilt:

- `student_<size>.pth` → `STUDENT_CLS[<size>]()`
- `teacher_<name>.pth` → `TEACHER_CLS[<name>]()`
- `student_kd_<student>_from_<teacher>.pth` → `STUDENT_CLS[<student>]()`
- `optuna_<size>.pth` and `optuna_kd_<student>_from_<teacher>.pth` → `STUDENT_CLS[<...>](**arch_kwargs)`

In [10]:
def load_checkpoint(ckpt_path):
    p = Path(ckpt_path)
    name = p.stem
    obj = torch.load(p, map_location=DEVICE, weights_only=False)
    arch_kwargs = {}
    if isinstance(obj, dict) and "state_dict" in obj:
        state, arch_kwargs = obj["state_dict"], obj.get("arch_kwargs", {})
    else:
        state = obj
    if name.startswith("optuna_kd_"):
        size = name[len("optuna_kd_"):].split("_from_")[0]
        model = STUDENT_CLS[size](**arch_kwargs)
    elif name.startswith("optuna_"):
        size = name[len("optuna_"):]
        model = STUDENT_CLS[size](**arch_kwargs)
    elif name.startswith("student_kd_"):
        size = name[len("student_kd_"):].split("_from_")[0]
        model = STUDENT_CLS[size]()
    elif name.startswith("student_"):
        model = STUDENT_CLS[name[len("student_"):]]()
    elif name.startswith("teacher_"):
        model = TEACHER_CLS[name[len("teacher_"):]]()
    else:
        raise ValueError(f"Unrecognized checkpoint name pattern: {name}")
    model.load_state_dict(state)
    return model.to(DEVICE).eval(), arch_kwargs

In [12]:
# === Configure ===
CHECKPOINT = CKPT_DIR / "student_nano.pth"   # change to any .pth in checkpoints/

# === Run ===
model, arch = load_checkpoint(CHECKPOINT)
print(f"loaded {CHECKPOINT.name}  params={sum(p.numel() for p in model.parameters()):,}")
if arch: print(f"arch_kwargs={arch}")

_, _, test_loader = get_loaders(Config(batch_size=8, device=DEVICE))
x, y = next(iter(test_loader))
x, y = x.to(DEVICE), y.to(DEVICE)
with torch.no_grad():
    probs = torch.sigmoid(model(x))
preds = (probs > 0.5).float()
for i in range(min(5, x.shape[0])):
    print(f"  sample {i}: true={y[i].tolist()}  pred={preds[i].tolist()}  probs={[f'{p:.2f}' for p in probs[i].tolist()]}")

FileNotFoundError: [Errno 2] No such file or directory: 'models\\CNN\\checkpoints\\student_nano.pth'

## 7. Evaluation

Runs each model on the test split and computes the team-standard multilabel metrics (`accuracy`, `f1_macro`, `auprc_macro`, `auroc_macro`, per-finger breakdowns) plus PR/ROC curves and per-finger confusion matrices using the peer's `evaluation/` module. Results land in `models/CNN/evaluations/<model_name>/`.

In [11]:
@torch.no_grad()
def collect_predictions(model, loader, device):
    model.eval()
    ps, ys = [], []
    for x, y in loader:
        ps.append(torch.sigmoid(model(x.to(device))).cpu())
        ys.append(y)
    return torch.cat(ps), torch.cat(ys)

def evaluate_checkpoint(ckpt_path, out_dir=None, save_per_finger=True):
    ckpt_path = Path(ckpt_path)
    name = ckpt_path.stem
    out_dir = Path(out_dir) if out_dir else EVAL_DIR / name
    out_dir.mkdir(parents=True, exist_ok=True)

    model, _ = load_checkpoint(ckpt_path)
    _, _, test_loader = get_loaders(Config(batch_size=64, device=DEVICE))
    probs, targets = collect_predictions(model, test_loader, torch.device(DEVICE))

    metrics = compute_multilabel_metrics(probs, targets)
    curves = compute_curves(probs, targets)
    save_metric_curves(curves, str(out_dir), save_per_finger=save_per_finger)
    save_confusion_matrices(probs, targets, str(out_dir))
    (out_dir / "metrics.json").write_text(json.dumps(metrics, indent=2))
    print(f"[{name}] acc={metrics['accuracy']:.3f}  f1={metrics['f1_macro']:.3f}  "
          f"auprc={metrics['auprc_macro']:.3f}  auroc={metrics['auroc_macro']:.3f}  -> {out_dir}")
    return metrics

### 7.1 Evaluate one model

In [ ]:
# === Configure ===
CHECKPOINT = CKPT_DIR / "teacher_resnet152.pth"

# === Run ===
evaluate_checkpoint(CHECKPOINT)

[teacher_resnet50] acc=0.570  f1=0.656  auprc=0.772  auroc=0.840  -> models\CNN\evaluations\teacher_resnet50


{'accuracy': 0.569672131147541,
 'finger_accuracy': 0.7879098360655739,
 'precision_macro': 0.7283980743398775,
 'recall_macro': 0.5988758782201404,
 'f1_macro': 0.656077525804717,
 'auprc_macro': 0.7717188810484209,
 'auroc_macro': 0.8401265001533746,
 'auprc_micro': 0.7693753814165893,
 'auroc_micro': 0.838561465127665,
 'per_finger': {'finger_0': {'precision': 0.7322033898302603,
   'recall': 0.5058548009366497,
   'f1': 0.5983379500900097,
   'auprc': 0.7207455317704257,
   'auroc': 0.7875421780286063},
  'finger_1': {'precision': 0.7564575645753666,
   'recall': 0.6721311475407633,
   'f1': 0.7118055555054826,
   'auprc': 0.8091403093450052,
   'auroc': 0.8706115169431481},
  'finger_2': {'precision': 0.7067669172929674,
   'recall': 0.6163934426227488,
   'f1': 0.6584938703528048,
   'auprc': 0.781319412370621,
   'auroc': 0.8565292809850724},
  'finger_3': {'precision': 0.7265624999997162,
   'recall': 0.6098360655735706,
   'f1': 0.66310160422822,
   'auprc': 0.7814598653916691

### 7.2 Evaluate every checkpoint in `checkpoints/`

Iterates all `.pth` files, evaluates each, and writes a top-level `summary.json` ranking them by `auprc_macro` (a sensible robust default for multi-label).

In [8]:
ckpts = sorted(CKPT_DIR.glob("*.pth"))
results = {}
for p in ckpts:
    try:
        results[p.stem] = evaluate_checkpoint(p)
    except Exception as e:
        print(f"[skip] {p.name}: {e}")

ranked = sorted(results.items(), key=lambda kv: kv[1].get("auprc_macro", float("-inf")), reverse=True)
summary = {name: {k: m.get(k) for k in ("accuracy", "f1_macro", "auprc_macro", "auroc_macro")} for name, m in ranked}
(EVAL_DIR / "summary.json").write_text(json.dumps(summary, indent=2))

print("\n========== RANKED BY auprc_macro ==========")
print(f"{'model':<48} {'acc':>6} {'f1':>6} {'auprc':>7} {'auroc':>7}")
for name, m in ranked:
    print(f"{name:<48} {m['accuracy']:>6.3f} {m['f1_macro']:>6.3f} {m['auprc_macro']:>7.3f} {m['auroc_macro']:>7.3f}")

[optuna_base] acc=0.575  f1=0.707  auprc=0.775  auroc=0.870  -> models\CNN\evaluations\optuna_base
[optuna_large] acc=0.593  f1=0.714  auprc=0.812  auroc=0.881  -> models\CNN\evaluations\optuna_large
[optuna_micro] acc=0.578  f1=0.697  auprc=0.793  auroc=0.871  -> models\CNN\evaluations\optuna_micro
[optuna_nano] acc=0.580  f1=0.698  auprc=0.758  auroc=0.855  -> models\CNN\evaluations\optuna_nano
[optuna_xlarge] acc=0.551  f1=0.690  auprc=0.726  auroc=0.830  -> models\CNN\evaluations\optuna_xlarge

========== RANKED BY auprc_macro ==========
model                                               acc     f1   auprc   auroc
optuna_large                                      0.593  0.714   0.812   0.881
optuna_micro                                      0.578  0.697   0.793   0.871
optuna_base                                       0.575  0.707   0.775   0.870
optuna_nano                                       0.580  0.698   0.758   0.855
optuna_xlarge                                     0.551  